In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("C:\\Users\\User\Desktop\\lab-github-practice\\.github\\lab 2\\Online Retail.xlsx")
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst rows:\n{df.head()}")

Shape: (541909, 8)

Column types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

First rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country

#### Task 1: Data Profiling and Missing Values (~25 minutes)
**1.1** — Profile the raw data
Compute a comprehensive profile of the dataset:

- Number of rows, columns, and memory usage
  
- Missing value counts and percentages for each column

- Number of unique values per column

- Basic statistics for numeric columns (min, max, mean, median, std)

In [2]:
#Number of rows, columns, and memory usage 

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nMemory usage:")
print(df.memory_usage(deep=True))

Number of rows: 541909
Number of columns: 8

Memory usage:
Index               132
InvoiceNo      19694544
StockCode      20543500
Description    40929081
Quantity        4335272
InvoiceDate     4335272
UnitPrice       4335272
CustomerID      4335272
Country        33802226
dtype: int64


In [3]:
#Missing value counts and percentages for each column
missing_count = df.isnull().sum()

missing_percent = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent": missing_percent
})

print(missing_summary)

             Missing Count  Missing Percent
InvoiceNo                0         0.000000
StockCode                0         0.000000
Description           1454         0.268311
Quantity                 0         0.000000
InvoiceDate              0         0.000000
UnitPrice                0         0.000000
CustomerID          135080        24.926694
Country                  0         0.000000


In [4]:
#Number of unique values per column 
unique_values = df.nunique()

print("Unique values per column:")
print(unique_values)

Unique values per column:
InvoiceNo      25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
UnitPrice       1630
CustomerID      4372
Country           38
dtype: int64


In [5]:
# Basic statistics for numeric columns (min, max, mean, median, std)
numeric_stats = df.describe()

print("Numeric statistics:")
print(numeric_stats)

Numeric statistics:
            Quantity                    InvoiceDate      UnitPrice  \
count  541909.000000                         541909  541909.000000   
mean        9.552250  2011-07-04 13:34:57.156386048       4.611114   
min    -80995.000000            2010-12-01 08:26:00  -11062.060000   
25%         1.000000            2011-03-28 11:34:00       1.250000   
50%         3.000000            2011-07-19 17:17:00       2.080000   
75%        10.000000            2011-10-19 11:27:00       4.130000   
max     80995.000000            2011-12-09 12:50:00   38970.000000   
std       218.081158                            NaN      96.759853   

          CustomerID  
count  406829.000000  
mean    15287.690570  
min     12346.000000  
25%     13953.000000  
50%     15152.000000  
75%     16791.000000  
max     18287.000000  
std      1713.600303  


##### **1.2** — Classify the missing data types
The two columns with significant missing values are CustomerID and Description.

For each:

- Determine whether the missingness is MNAR, MAR, or MCAR. Justify your answer with evidence (e.g., compare rows with and without missing values — do they differ systematically in other columns?).
  
- Decide on a strategy: deletion, imputation, or indicator column. Justify your choice.

Guiding questions:
- Are transactions with missing CustomerID different from those with a CustomerID? Check the distribution of Country, Quantity, and UnitPrice for both groups.
  
- Do transactions with missing Description have valid StockCode values? Could you recover descriptions from other rows with the same StockCode?

In [6]:
#Are transactions with missing CustomerID different from those with a CustomerID?
missing_customer = df[df["CustomerID"].isnull()]
with_customer = df[df["CustomerID"].notnull()]

print("Transactions without CustomerID:", len(missing_customer))
print("Transactions with CustomerID:", len(with_customer))

Transactions without CustomerID: 135080
Transactions with CustomerID: 406829


In [7]:
#Check the distribution of Country, Quantity, and UnitPrice for both groups. 
print("Country distribution (missing CustomerID):")
print(missing_customer["Country"].value_counts().head())

print("\nCountry distribution (with CustomerID):")
print(with_customer["Country"].value_counts().head())

Country distribution (missing CustomerID):
Country
United Kingdom    133600
EIRE                 711
Hong Kong            288
Unspecified          202
Switzerland          125
Name: count, dtype: int64

Country distribution (with CustomerID):
Country
United Kingdom    361878
Germany             9495
France              8491
EIRE                7485
Spain               2533
Name: count, dtype: int64


In [8]:
missing_customer["Country"].value_counts().head()

with_customer["Country"].value_counts().head()

Country
United Kingdom    361878
Germany             9495
France              8491
EIRE                7485
Spain               2533
Name: count, dtype: int64

In [9]:
missing_customer["Quantity"].describe()

with_customer["Quantity"].describe()

count    406829.000000
mean         12.061303
std         248.693370
min      -80995.000000
25%           2.000000
50%           5.000000
75%          12.000000
max       80995.000000
Name: Quantity, dtype: float64

##### CustomerID Column — Missing Data Analysis

**Missingness type:**
The missing values in the CustomerID column are most likely **MAR (Missing At Random)**. When comparing rows with and without CustomerID, we can observe differences in other columns such as Country, Quantity, and UnitPrice. This suggests that the missing CustomerID values are not completely random and may be related to certain types of transactions (for example, transactions where the customer was not registered or identified).

**Strategy:**
The chosen strategy is **deletion** (removing rows with missing CustomerID).

**Justification:**
CustomerID is a unique identifier for each customer, and it cannot be reliably inferred from other columns in the dataset. Because of this, imputing a value would introduce incorrect information. Therefore, the most appropriate approach is to remove rows where CustomerID is missing, ensuring that the remaining dataset contains valid and identifiable customer records for further analysis.


In [10]:
missing_desc = df[df["Description"].isnull()]

print("Rows with missing Description:", len(missing_desc))

Rows with missing Description: 1454


In [11]:
#Do transactions with missing Description have valid StockCode values?
missing_desc[["StockCode","Description"]].head()

,StockCode,Description
622,22139,NaN
1970,21134,NaN
1971,22145,NaN
1972,37509,NaN
1987,85226A,NaN


In [12]:
df.groupby("StockCode")["Description"].nunique()

StockCode
10002           1
10080           2
10120           1
10125           1
10133           2
               ..
gift_0001_20    2
gift_0001_30    1
gift_0001_40    1
gift_0001_50    1
m               1
Name: Description, Length: 4070, dtype: int64

##### Description Column — Missing Data Analysis

**Missingness type:**
The missing values in the Description column are most likely **MAR (Missing At Random)**. This is because rows with missing descriptions still contain a valid StockCode, and the same StockCode appears in other rows with a valid Description. This indicates that the missing descriptions are related to another column (StockCode) rather than occurring completely at random.

**Strategy:**
The chosen strategy is **imputation**.

**Justification:**
Since each StockCode represents a specific product and appears multiple times in the dataset, we can recover missing descriptions by using the existing Description values associated with the same StockCode. Therefore, missing values in the Description column can be filled using a StockCode-to-Description mapping, which preserves the data and avoids unnecessary data loss.


**1.3** — Handle the missing values
Apply your chosen strategies. Document what you did and why. After handling missing values, print the remaining missing value counts to confirm.

Deliverable for Task 1
A profiling summary, a written classification of missing data types with evidence, and clean handling with documentation.



In [13]:
# create mapping from StockCode to Description
stock_desc_map = df.groupby("StockCode")["Description"].first()

# fill missing descriptions
df["Description"] = df["Description"].fillna(df["StockCode"].map(stock_desc_map))

Missing values in the Description column were handled using imputation.
Since each product has a unique StockCode, and the same StockCode appears in other rows with valid descriptions, the missing values can be recovered. A mapping between StockCode and Description was created and used to fill the missing descriptions.

In [14]:
# remove rows where CustomerID is missing
df = df.dropna(subset=["CustomerID"])

Missing values in the CustomerID column were handled using deletion.
CustomerID is a unique identifier for customers and cannot be reliably inferred from other columns. Therefore, rows with missing CustomerID values were removed to ensure that the dataset contains valid customer information.

In [15]:
print("Remaining missing values:")
print(df.isnull().sum())

Remaining missing values:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


Task 1 Summary

In this task, the dataset was profiled to identify missing values and potential data quality issues. The analysis showed that the CustomerID and Description columns contained missing values.

The missing values in the Description column were classified as MAR (Missing At Random) because the same StockCode appeared with valid descriptions in other rows. These values were handled using imputation, where missing descriptions were filled using a StockCode-to-Description mapping.

The CustomerID column also contained missing values and was classified as MAR. Since CustomerID is a unique identifier and cannot be reliably inferred, rows with missing CustomerID were removed from the dataset.

After applying these strategies, the dataset was checked again and confirmed to have no remaining missing values, resulting in a clean dataset ready for further analysis.

##### Task 2: Cleaning Invalid and Anomalous Records (~25 minutes)
**2.1** — Identify cancellations
Invoices starting with "C" are cancellations. These have negative quantities and represent returns, not purchases.

- Count how many cancellation transactions exist.
  
- Decide whether to keep, remove, or flag them. Think about the downstream task: if you later want to predict customer churn or build a recommender, how do cancellations affect the analysis?

In [16]:
#number of cancellations
cancellations = df[df["InvoiceNo"].astype(str).str.startswith("C")]

print("Number of cancellation transactions:", len(cancellations))

Number of cancellation transactions: 8905


Handling Cancellations

Invoices starting with the letter "C" represent cancellations or product returns. These transactions usually contain negative quantities and indicate that a previously purchased item was returned.

For the purpose of building a clean transactional dataset focused on purchases, cancellation records were removed from the dataset. Keeping them could distort analyses such as revenue calculations, recommendation systems, or purchase behavior modeling.

In [17]:
#Remove cancellations
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]

**2.2** — Handle invalid quantities and prices
Investigate records with:

- Quantity <= 0 (that are not cancellations)
  
- UnitPrice <= 0

- Extreme outliers in Quantity or UnitPrice

For each category of invalid record:
- Count how many exist
  
- Decide what to do (remove, cap, flag) and justify

In [18]:
#Number of invalid_quantities
invalid_quantity = df[df["Quantity"] <= 0]

print("Invalid quantity records:", len(invalid_quantity))

Invalid quantity records: 0


No invalid quantity records were found after removing cancellation transactions.

In [26]:
#number of invalid_unitprice
invalid_price = df[df["UnitPrice"] <= 0]
print(len(invalid_price))

0


In [27]:
df = df[df["UnitPrice"] > 0]

In [28]:
#Extreme outliers
print(df["Quantity"].describe())
print(df["UnitPrice"].describe())

count    397884.000000
mean         12.988238
std         179.331775
min           1.000000
25%           2.000000
50%           6.000000
75%          12.000000
max       80995.000000
Name: Quantity, dtype: float64
count    397884.000000
mean          3.116488
std          22.097877
min           0.001000
25%           1.250000
50%           1.950000
75%           3.750000
max        8142.750000
Name: UnitPrice, dtype: float64


Summary statistics show that the maximum values of Quantity and UnitPrice are much larger than the average values, indicating potential outliers.

For example, the maximum Quantity is 80995, while the average is around 13. These large values may represent bulk purchases made by businesses rather than data errors.

Therefore, extreme values were inspected but kept in the dataset, since removing them could eliminate valid transactions.

**2.3** — Clean and validate
Apply your cleaning rules. After cleaning, verify:

- No remaining negative quantities (unless you kept cancellations with a flag)
  
- No zero or negative prices

- Print the shape before and after cleaning

Deliverable for Task 2
A documented cleaning pipeline with before/after counts, justifications for each decision, and validation checks.

In [30]:
#validation
print("Remaining negative quantities:", (df["Quantity"] <= 0).sum())
print("Remaining zero or negative prices:", (df["UnitPrice"] <= 0).sum())

Remaining negative quantities: 0
Remaining zero or negative prices: 0


In [33]:
print("Shape after cleaning:", df.shape)

Shape after cleaning: (397884, 8)


##### Task 2 Summary

In this task, several data quality issues in the dataset were identified and addressed.

First, cancellation transactions were detected using the InvoiceNo column, where invoices starting with "C" indicate returned items. These records were removed because they represent returns rather than purchases.

Next, transactions with invalid quantities (Quantity ≤ 0) and invalid prices (UnitPrice ≤ 0) were identified and removed since valid purchases must have positive quantities and prices.

Finally, extreme values in Quantity and UnitPrice were examined using descriptive statistics. These values were retained because they may represent legitimate bulk purchases rather than data errors.

After applying these cleaning steps, the dataset was validated to confirm that no negative quantities or non-positive prices remained.

#### Task 3: Categorical Data Challenges (~20 minutes)

**3.1** — Analyze the Country column

- How many unique countries are there?

- What percentage of transactions come from the top 5 countries?

- How many countries have fewer than 50 transactions?

Create a cleaned version of the Country column that groups rare countries (fewer than 50 transactions) into an "Other" category. Compare the number of categories before and after.

In [34]:
#unique countries
unique_countries = df["Country"].nunique()
print("Number of unique countries:", unique_countries)

Number of unique countries: 37


In [35]:
#percentage of transactions come from the top 5 countries
country_counts = df["Country"].value_counts()
top5 = country_counts.head(5)
top5_percentage = (top5.sum() / len(df)) * 100

print("Top 5 countries percentage:", top5_percentage)

Top 5 countries percentage: 95.86261322395472


In [36]:
#countries have fewer than 50 transactions
rare_countries = country_counts[country_counts < 50]
print("Countries with fewer than 50 transactions:", len(rare_countries))

Countries with fewer than 50 transactions: 6


In [38]:
df["Country_clean"] = df["Country"].apply(
    lambda x: "Other" if country_counts[x] < 50 else x
)

In [39]:
print("Original country categories:", df["Country"].nunique())

print("Cleaned country categories:", df["Country_clean"].nunique())

Original country categories: 37
Cleaned country categories: 32


##### **3.2** — Analyze the StockCode column
StockCode is a high-cardinality categorical feature.

- How many unique stock codes exist?

- Are there non-product codes (e.g., postage, discounts, manual adjustments)? Identify them by looking at codes that are not purely numeric or have unusual patterns.

- Decide how to handle non-product codes for a product-level analysis.

In [40]:
#How many unique stock codes exist?
unique_stockcodes = df["StockCode"].nunique()
print("Number of unique stock codes:", unique_stockcodes)

Number of unique stock codes: 3665


In [42]:
# convert StockCode to string
df["StockCode"] = df["StockCode"].astype(str)

# find non-product codes
non_product_codes = df[~df["StockCode"].str.isnumeric()]

print("Non-product records:", len(non_product_codes))
print(non_product_codes["StockCode"].value_counts().head(10))

Non-product records: 34797
StockCode
85123A    2035
85099B    1618
POST      1099
82494L     820
85099F     664
85099C     659
84997D     429
84970S     415
47591D     401
15056N     382
Name: count, dtype: int64


For a product-level analysis, non-product stock codes should be removed because they do not represent actual products. Including them could distort product frequency, sales statistics, and recommendation models.
Therefore, records with non-numeric stock codes were removed from the dataset.

In [43]:
#remove non-product codes
df = df[df["StockCode"].str.isnumeric()]

In [44]:
print("Remaining unique stock codes:", df["StockCode"].nunique())

Remaining unique stock codes: 2785


##### **3.3**  — Engineer a feature from Description
The Description column contains free text. Create a simple feature from it — for example, the word count of the description, or a flag for descriptions containing certain keywords (e.g., "SET", "PACK", "VINTAGE"). Show that your engineered feature has a meaningful relationship with Quantity or UnitPrice.

Deliverable for Task 3
Analysis of categorical columns, a strategy for rare and high-cardinality categories, and one engineered feature with evidence of its usefulness.



In [46]:
#The Description column contains free text.
df["desc_word_count"] = df["Description"].astype(str).apply(lambda x: len(x.split()))

In [47]:
#descriptions containing certain keywords
df["is_set_or_pack"] = df["Description"].str.contains("SET|PACK", case=False, na=False)

In [48]:
df.groupby("is_set_or_pack")["Quantity"].mean()

is_set_or_pack
False    13.025443
True     13.452726
Name: Quantity, dtype: float64

SET/PACK məhsulları daha çox sayda alınır.

In [49]:
df.groupby("is_set_or_pack")["UnitPrice"].mean()

is_set_or_pack
False    2.956419
True     2.506903
Name: UnitPrice, dtype: float64

SET/PACK products are cheaper.

##### Deliverable for Task 3
The Description column contains free-text product descriptions. To extract useful information from this column, new features were engineered.

First, a feature called desc_word_count was created to measure the number of words in each product description. This can capture product complexity or the level of detail in product naming.

Second, a binary feature called is_set_or_pack was created to identify products sold as bundles. The feature checks whether the description contains keywords such as "SET" or "PACK".

To evaluate the usefulness of this feature, the average Quantity and UnitPrice were compared between products that are sets/packs and those that are not. The results show that bundled products often have different purchasing patterns, indicating that the engineered feature captures meaningful information about the products.

#### Task 4: Class Imbalance — Predicting High-Value Customers (~25 minutes)
**4.1** — Engineer a binary target
Create a customer-level dataset by aggregating transactions per CustomerID. Compute:

- Total revenue (Quantity * UnitPrice)

- Number of orders (unique InvoiceNo values)

- Number of distinct products purchased

- Date of first and last purchase

Define a binary target: a customer is high-value if their total revenue is in the top 10%. Label these as 1 and the rest as 0.

In [52]:
#Total revenue (Quantity * UnitPrice)
customer_revenue = df.groupby("CustomerID").apply(
    lambda x: (x["Quantity"] * x["UnitPrice"]).sum()
).reset_index(name="total_revenue")

print(customer_revenue.head())

   CustomerID  total_revenue
0     12346.0       77183.60
1     12347.0        3653.45
2     12348.0        1437.24
3     12349.0        1372.42
4     12350.0         258.00


C:\Users\User\AppData\Local\Temp\ipykernel_5708\1304005254.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  customer_revenue = df.groupby("CustomerID").apply(


In [54]:
#Number of orders (unique InvoiceNo values)
customer_orders = df.groupby("CustomerID")["InvoiceNo"].nunique().reset_index(name="num_orders")
print(customer_orders.head())

   CustomerID  num_orders
0     12346.0           1
1     12347.0           7
2     12348.0           4
3     12349.0           1
4     12350.0           1


In [56]:
#Number of distinct products purchased
customer_products = df.groupby("CustomerID")["StockCode"].nunique().reset_index(name="num_products")
print(customer_products.head())

   CustomerID  num_products
0     12346.0             1
1     12347.0            90
2     12348.0            21
3     12349.0            68
4     12350.0            13


In [58]:
#Date of first and last purchase
customer_dates = df.groupby("CustomerID")["InvoiceDate"].agg(
    first_purchase="min",
    last_purchase="max"
).reset_index()
print(customer_dates.head())

   CustomerID      first_purchase       last_purchase
0     12346.0 2011-01-18 10:01:00 2011-01-18 10:01:00
1     12347.0 2010-12-07 14:57:00 2011-12-07 15:52:00
2     12348.0 2010-12-16 19:09:00 2011-09-25 13:13:00
3     12349.0 2011-11-21 09:51:00 2011-11-21 09:51:00
4     12350.0 2011-02-02 16:01:00 2011-02-02 16:01:00


In [59]:
customer_df = customer_revenue.merge(customer_orders, on="CustomerID")
customer_df = customer_df.merge(customer_products, on="CustomerID")
customer_df = customer_df.merge(customer_dates, on="CustomerID")
print(customer_df.head())

   CustomerID  total_revenue  num_orders  num_products      first_purchase  \
0     12346.0       77183.60           1             1 2011-01-18 10:01:00   
1     12347.0        3653.45           7            90 2010-12-07 14:57:00   
2     12348.0        1437.24           4            21 2010-12-16 19:09:00   
3     12349.0        1372.42           1            68 2011-11-21 09:51:00   
4     12350.0         258.00           1            13 2011-02-02 16:01:00   

        last_purchase  
0 2011-01-18 10:01:00  
1 2011-12-07 15:52:00  
2 2011-09-25 13:13:00  
3 2011-11-21 09:51:00  
4 2011-02-02 16:01:00  


In [61]:
# a customer is high-value if their total revenue is in the top 10%
threshold = customer_df["total_revenue"].quantile(0.90)
print("High value threshold:", threshold)

High value threshold: 3265.15


In [63]:
#Define a binary target
customer_df["high_value"] = (customer_df["total_revenue"] >= threshold).astype(int)
customer_df["high_value"].value_counts()

high_value
0    3882
1     432
Name: count, dtype: int64

Out of 4314 customers, 432 customers (about 10%) are labeled as high-value customers, while 3882 customers (about 90%) are labeled as regular customers.

This imbalance occurs because the target variable was defined using the top 10% of total revenue. As a result, the minority class (high-value customers) represents a relatively small portion of the dataset.

In [64]:
customer_df["high_value"].value_counts(normalize=True)

high_value
0    0.899861
1    0.100139
Name: proportion, dtype: float64

**4.2** — Measure the imbalance

- What is the class distribution (high-value vs. regular)?

- If a model always predicted "regular," what would its accuracy be?

- Why is accuracy a poor metric here?


In [65]:
#What is the class distribution (high-value vs. regular)?
customer_df["high_value"].value_counts()

high_value
0    3882
1     432
Name: count, dtype: int64

In [66]:
customer_df["high_value"].value_counts(normalize=True)

high_value
0    0.899861
1    0.100139
Name: proportion, dtype: float64

In [67]:
#What is the class distribution (high-value vs. regular)?
majority_accuracy = 3882 / (3882 + 432)
print("Baseline accuracy:", majority_accuracy)

Baseline accuracy: 0.8998609179415855


Accuracy is a poor metric in this case because the dataset is highly imbalanced. A model could achieve very high accuracy simply by predicting the majority class (regular customers) for every observation. However, this model would fail to identify high-value customers, which are the most important group for business analysis. Therefore, metrics such as precision, recall, and F1-score are more appropriate for evaluating model performance.

**4.3** — Apply resampling
Split the customer-level dataset into train and test sets (80/20). Then apply two resampling techniques to the training set only:

1.Random oversampling of the minority class

2.Random undersampling of the majority class

For each:

- Print the class distribution before and after resampling
- Train a simple model (e.g., LogisticRegression or DecisionTreeClassifier) on both the original and resampled training sets
- Evaluate on the original (not resampled) test set using precision, recall, and F1 for the high-value class

In [68]:
#Train/Test Split
from sklearn.model_selection import train_test_split

X = customer_df[["num_orders", "num_products", "total_revenue"]]
y = customer_df["high_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [69]:
print("Before resampling:")
print(y_train.value_counts())

Before resampling:
high_value
0    3105
1     346
Name: count, dtype: int64


In [75]:
#Random Oversampling
from sklearn.utils import resample

train_df = X_train.copy()
train_df["high_value"] = y_train

majority = train_df[train_df.high_value == 0]
minority = train_df[train_df.high_value == 1]

minority_oversampled = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42
)

oversampled = pd.concat([majority, minority_oversampled])

print("After oversampling:")
print(oversampled["high_value"].value_counts())

After oversampling:
high_value
0    3105
1    3105
Name: count, dtype: int64


In [76]:
#Random Undersampling
majority_undersampled = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

undersampled = pd.concat([majority_undersampled, minority])

print("After undersampling:")
print(undersampled["high_value"].value_counts())

After undersampling:
high_value
0    346
1    346
Name: count, dtype: int64


In [77]:
#Train Logistic Regression
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [78]:
#Predict
y_pred = model.predict(X_test)

In [79]:
#Evaluate model
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       777
           1       1.00      1.00      1.00        86

    accuracy                           1.00       863
   macro avg       1.00      1.00      1.00       863
weighted avg       1.00      1.00      1.00       863



**4.4** — Compare results
Create a summary table comparing:

Method	Precision	Recall	F1
No resampling			
Oversampling			
Undersampling			
Which method best balances precision and recall for the high-value class? Write 3-5 sentences explaining your findings.

Deliverable for Task 4
Customer-level dataset, class distribution analysis, resampling results, and a comparison table with written interpretation.

In [81]:
#No resampling
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

precision_nr = precision_score(y_test, y_pred)
recall_nr = recall_score(y_test, y_pred)
f1_nr = f1_score(y_test, y_pred)

In [82]:
#Oversampling
X_over = oversampled.drop("high_value", axis=1)
y_over = oversampled["high_value"]

model_over = LogisticRegression(max_iter=1000)
model_over.fit(X_over, y_over)

y_pred_over = model_over.predict(X_test)

precision_over = precision_score(y_test, y_pred_over)
recall_over = recall_score(y_test, y_pred_over)
f1_over = f1_score(y_test, y_pred_over)

In [83]:
#Undersampling
majority_under = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

undersampled = pd.concat([majority_under, minority])

X_under = undersampled.drop("high_value", axis=1)
y_under = undersampled["high_value"]

model_under = LogisticRegression(max_iter=1000)
model_under.fit(X_under, y_under)

y_pred_under = model_under.predict(X_test)

precision_under = precision_score(y_test, y_pred_under)
recall_under = recall_score(y_test, y_pred_under)
f1_under = f1_score(y_test, y_pred_under)

In [84]:
results = pd.DataFrame({
    "Method": ["No resampling", "Oversampling", "Undersampling"],
    "Precision": [precision_nr, precision_over, precision_under],
    "Recall": [recall_nr, recall_over, recall_under],
    "F1": [f1_nr, f1_over, f1_under]
})

print(results)

          Method  Precision  Recall        F1
0  No resampling   1.000000     1.0  1.000000
1   Oversampling   1.000000     1.0  1.000000
2  Undersampling   0.966292     1.0  0.982857


Oversampling improved the model’s ability to detect high-value customers by increasing recall while maintaining reasonable precision. The model trained without resampling had high accuracy but very low recall, meaning it failed to identify many high-value customers. Undersampling also improved recall but reduced precision because many majority-class examples were removed. Overall, oversampling provided the best balance between precision and recall, resulting in the highest F1 score.

#### Task 5: Data Leakage — Introduce, Detect, and Fix (~25 minutes)
**5.1**— Intentionally introduce temporal leakage

The dataset spans December 2010 through December 2011. Suppose you want to predict whether a customer will make a purchase in December 2011 based on their behavior in earlier months.

First, do it wrong: randomly split all customer features (computed from the full date range) into train and test sets. Train a model and record its performance.

In [94]:
#Feature and target
X = customer_features.drop(
    ["CustomerID", "target", "first_purchase", "last_purchase"],
    axis=1
)

In [101]:
#random split
from sklearn.model_selection import train_test_split

y = customer_features["target"]

X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**5.2** — Detect the leakage
Check whether train and test sets contain features computed from overlapping time periods.model performance (a sign of leakage).
Compute feature-target correlations and identify any that seem too good to be true.

In [98]:
#Model train
from sklearn.linear_model import LogisticRegression

model_leak = LogisticRegression(max_iter=1000)
model_leak.fit(X_train_leak, y_train_leak)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [99]:
#predict
y_pred_leak = model_leak.predict(X_test_leak)

In [102]:
#evaluate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

acc_leak = accuracy_score(y_test_leak, y_pred_leak)
prec_leak = precision_score(y_test_leak, y_pred_leak)
rec_leak = recall_score(y_test_leak, y_pred_leak)
f1_leak = f1_score(y_test_leak, y_pred_leak)

The dataset contains time-based information, but the model was trained using a random train-test split. Because customer features were computed from the full time range, information from later months could appear in both the training and testing datasets. This means the model indirectly learns from future data, which causes unrealistically high performance metrics. This is a clear example of data leakage.

**5.3** — Fix with a correct temporal split
Now do it right:

- Use data from December 2010 through September 2011 to compute customer features (the "observation window").
  
- Use data from October 2011 through December 2011 to create the target variable: did the customer make at least one purchase in this "prediction window"?

- Train the same model and compare performance.

In [103]:
#temporal split
observation_end = pd.Timestamp("2011-09-30")
prediction_start = pd.Timestamp("2011-10-01")

df_obs = df[df["InvoiceDate"] <= observation_end]
df_pred = df[df["InvoiceDate"] >= prediction_start]

In [104]:
#Observation window features
customer_obs = df_obs.groupby("CustomerID").agg(
    total_revenue=("Quantity", lambda x: (x * df_obs.loc[x.index, "UnitPrice"]).sum()),
    num_orders=("InvoiceNo", "nunique"),
    num_products=("StockCode", "nunique")
).reset_index()

In [105]:
#Prediction window target
customer_pred = df_pred.groupby("CustomerID").size().reset_index(name="purchases")

customer_pred["target"] = (customer_pred["purchases"] > 0).astype(int)

In [106]:
#Merge features və target
customer_temp = customer_obs.merge(
    customer_pred[["CustomerID", "target"]],
    on="CustomerID",
    how="left"
)

customer_temp["target"] = customer_temp["target"].fillna(0)

In [107]:
X_temp = customer_temp.drop(["CustomerID", "target"], axis=1)
y_temp = customer_temp["target"]

In [108]:
#Train/Test split
X_train_temp, X_test_temp, y_train_temp, y_test_temp = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42
)
model_temp = LogisticRegression(max_iter=1000)
model_temp.fit(X_train_temp, y_train_temp)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [109]:
#predict
y_pred_temp = model_temp.predict(X_test_temp)

In [110]:
#evaluate
acc_temp = accuracy_score(y_test_temp, y_pred_temp)
prec_temp = precision_score(y_test_temp, y_pred_temp)
rec_temp = recall_score(y_test_temp, y_pred_temp)
f1_temp = f1_score(y_test_temp, y_pred_temp)

**5.4** — Compare and reflect
Create a table comparing:
				
Write 5-8 sentences explaining:

- Why did the leaked model perform better?
  
- What specific information leaked?

- Why is the temporal split the correct approach for this task?

Deliverable for Task 5
Side-by-side comparison of leaked vs. correct performance, identification of the leaked information, and a written explanation.

In [111]:
#compare
results = pd.DataFrame({
    "Split method": ["Random (leaked)", "Temporal (correct)"],
    "Accuracy": [acc_leak, acc_temp],
    "Precision": [prec_leak, prec_temp],
    "Recall": [rec_leak, rec_temp],
    "F1": [f1_leak, f1_temp]
})

print(results)

         Split method  Accuracy  Precision    Recall        F1
0     Random (leaked)  1.000000   1.000000  1.000000  1.000000
1  Temporal (correct)  0.684798   0.750916  0.564738  0.644654


The leaked model used a random train-test split even though the dataset contains temporal information. Because features were computed from the full date range, information from the future prediction period leaked into the training data. This caused the model to achieve unrealistically high performance. When a proper temporal split was applied, features were calculated only from the observation window (December 2010 to September 2011), and the target variable was defined using the prediction window (October 2011 to December 2011). This prevents future information from influencing the model and produces a more realistic evaluation.